# 00 — Environment Setup Verification

**Phase:** Phase 1 — Environment Setup + Data Preparation

**Purpose:** verify that the GPU/CUDA/library environment required for the QLoRA fine-tuning
phases (3+) is actually present and functional, and confirm that the target model's
config and processor load correctly from the Hugging Face Hub.

**IMPORTANT — Colab vs. local distinction:**

This notebook's GPU/CUDA checks are **authoritative only when this notebook is executed on
Google Colab with a GPU runtime**. Per `AGENTS.md` / `CLAUDE.md` §12–13, GPU-dependent
behavior (CUDA availability, device name, VRAM) must be validated on the actual target
execution environment (Colab), not inferred from a local machine.

If this notebook happens to be opened and executed on a local, CPU-only machine (as it was
for the initial authoring/validation pass recorded in this file), the results below are
useful only for **structural validation** — i.e. confirming the notebook runs top-to-bottom
without raising an exception and that the non-GPU library/config/processor checks pass. A
local run reporting `cuda available: False` is an **expected, honest, non-failing result**
on a machine with no CUDA device — it is NOT evidence that the environment is broken, and it
does NOT substitute for real Colab GPU validation. Do not treat a local run of this notebook
as satisfying any GPU-dependent acceptance criterion in `docs/EXPERIMENT_SPEC.md` or
`docs/IMPLEMENTATION_PLAN.md`.

In [1]:
# Detect whether this notebook is running on Google Colab.
# This gates the Colab-only setup cell below and lets every later cell report
# results honestly rather than silently assuming one environment or the other.
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running on Google Colab: {IN_COLAB}")
print(f"'google.colab' in sys.modules: {'google.colab' in sys.modules}")

Running on Google Colab: True
'google.colab' in sys.modules: True


## Colab-only repository setup

Colab starts each session with a fresh VM, so the repository and this project's package
(`vlm_lab`) are not present until explicitly obtained and installed. This step is skipped
entirely when `IN_COLAB` is `False` (e.g. on this local machine, where the package is
already installed editable in `.venv`), so re-running this notebook locally never tries to
`git clone` or reinstall anything.

Two ways to get the repository onto the Colab VM:

1. **`git clone`** (default below) — `REPO_URL` is set to this repository's GitHub remote,
   and `GIT_REF` pins the branch to check out (currently the pull-request branch this Phase 1
   work lives on, since it has not merged to the default branch yet — see the `TODO` in the
   next cell). The cell is idempotent: if `repo/` already exists on disk (e.g. after a kernel
   restart on the same Colab VM, which does not wipe the filesystem), it fetches and checks
   out `GIT_REF` again and pulls instead of re-cloning, so **Restart & Run All** works
   correctly on a second pass.
2. **Upload / Google Drive** — alternatively, upload the repo as a zip via the Colab file
   browser, or mount Google Drive (`google.colab.drive.mount`) if the repo already lives
   there, then `%cd` into it. Drive mounting is **not** a hard requirement of this notebook —
   it is just one option for getting the code onto the VM.

In [2]:
REPO_URL = "https://github.com/Mr-Kondo/finetuning_vlm.git"

# TODO: this repository's Phase 1 work currently lives on a pull-request branch,
# not on the default branch (see docs/STATE.md — PR #1 is open, not yet merged).
# A plain `git clone` without a branch would check out `main`, which does not
# yet contain notebooks/, src/vlm_lab/, etc. Once PR #1 merges, set GIT_REF back
# to "" so this clones the default branch instead of a branch that will
# eventually be deleted.
GIT_REF = "main"  # "" = default branch

if IN_COLAB:
    import importlib
    import os
    import subprocess
    import sys

    if not REPO_URL:
        raise RuntimeError(
            "IN_COLAB is True but REPO_URL is not set. Set REPO_URL above before "
            "running this notebook on Colab — later cells (model config / processor "
            "loading) depend on `vlm_lab` being installed, so continuing without a "
            "repository would leave the notebook in a silently broken state."
        )

    repo_dir = "finetuning_vlm"

    if os.path.isdir(repo_dir):
        print(f"'{repo_dir}' already exists — updating instead of re-cloning.")
        subprocess.run(["git", "-C", repo_dir, "fetch", "origin"], check=True)
        if GIT_REF:
            subprocess.run(["git", "-C", repo_dir, "checkout", GIT_REF], check=True)
        subprocess.run(["git", "-C", repo_dir, "pull"], check=True)
    else:
        clone_cmd = ["git", "clone"]
        if GIT_REF:
            clone_cmd += ["--branch", GIT_REF]
        clone_cmd += [REPO_URL, repo_dir]
        subprocess.run(clone_cmd, check=True)

    # Change to repo directory and install
    os.chdir(os.path.abspath(repo_dir))

    # Add current directory to sys.path so the editable install is visible
    if os.getcwd() not in sys.path:
        sys.path.append(os.getcwd())

    install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
        capture_output=True, text=True
    )

    if install.returncode != 0:
        raise RuntimeError(f"pip install failed:\n{install.stderr}")

    # Invalidate caches to ensure the new module is found
    importlib.invalidate_caches()

    try:
        importlib.import_module("vlm_lab")
    except ImportError as exc:
        # Check if the package is under a src/ directory
        src_path = os.path.join(os.getcwd(), "src")
        if os.path.isdir(src_path) and src_path not in sys.path:
            sys.path.append(src_path)
            importlib.invalidate_caches()
            try:
                importlib.import_module("vlm_lab")
            except ImportError:
                raise RuntimeError(f"Failed to import vlm_lab even after adding src/ to path: {exc}")
        else:
            raise RuntimeError(f"import vlm_lab failed: {exc}")

    print("Repository cloned, package installed, and `import vlm_lab` succeeded.")
else:
    print("Not running on Colab — skipping repo clone / install.")

'finetuning_vlm' already exists — updating instead of re-cloning.
Repository cloned, package installed, and `import vlm_lab` succeeded.


## Python and core library versions

Later phases (QLoRA training, evaluation) depend on specific behavior of `torch`,
`transformers`, `datasets`, and `Pillow`. Recording the actually-installed versions here
(rather than assuming versions from `pyproject.toml`'s lower bounds) is part of the
reproducibility record required by `CLAUDE.md` §11.

In [3]:
import platform

import torch
import transformers
import datasets
import PIL

print(f"Python version: {platform.python_version()}")
print(f"torch version: {torch.__version__}")
print(f"transformers version: {transformers.__version__}")
print(f"datasets version: {datasets.__version__}")
print(f"Pillow (PIL) version: {PIL.__version__}")

Python version: 3.12.13
torch version: 2.13.0+cu130
transformers version: 5.15.0
datasets version: 5.0.1
Pillow (PIL) version: 10.4.0


## CUDA availability

4-bit QLoRA training requires a CUDA GPU. This cell must never raise, whether or not a
CUDA device is present, so that it behaves correctly both on this local CPU-only machine and
on a Colab GPU runtime.

In [4]:
cuda_available = torch.cuda.is_available()
print(f"torch.cuda.is_available(): {cuda_available}")

if cuda_available:
    print(f"torch.version.cuda: {torch.version.cuda}")
    print(f"CUDA device count: {torch.cuda.device_count()}")
else:
    print("No CUDA device available: torch.version.cuda and device count are not applicable.")

torch.cuda.is_available(): True
torch.version.cuda: 13.0
CUDA device count: 1


## GPU identity and memory (if available)

Knowing the actual assigned GPU (name, total VRAM) is required later to reason about
whether 4-bit QLoRA fine-tuning of a 4B-parameter VLM will fit in memory (see
`docs/DECISIONS.md` ADR-014, the production-shape VRAM go/no-go gate). This cell reports
a clear message instead of crashing when no CUDA device is present.

In [5]:
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    total_memory_gib = props.total_memory / (1024 ** 3)
    print(f"GPU name: {device_name}")
    print(f"Total GPU memory: {total_memory_gib:.2f} GiB")
else:
    device_name = None
    total_memory_gib = None
    print("No CUDA device available: skipping GPU name / memory query.")

GPU name: Tesla T4
Total GPU memory: 14.56 GiB


## Target model config

Loading only the model **config** (a small JSON file from the Hub) confirms that the model
identifier is correct and reachable, and that the installed `transformers` version
recognizes its architecture — without downloading the multi-gigabyte model weights, which
belongs to Phase 3+. `AutoConfig` (rather than a `Qwen3VL*`-specific class) is used for
forward-compatible, standard loading.

In [6]:
from transformers import AutoConfig

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

model_config = AutoConfig.from_pretrained(MODEL_ID)

print(f"Config class: {type(model_config).__name__}")
print(f"model_type: {getattr(model_config, 'model_type', None)}")
print(f"architectures: {getattr(model_config, 'architectures', None)}")

Config class: Qwen3VLConfig
model_type: qwen3_vl
architectures: ['Qwen3VLForConditionalGeneration']


## Target model processor

The processor (tokenizer + image processor) is also lightweight relative to the model
weights and is required for every later phase that builds prompts or preprocesses images.
Loading it here confirms the processor files are present on the Hub and compatible with the
installed `transformers` version, using the standard `AutoProcessor` entry point.

In [7]:
import os
import sys
import importlib

def run_alignment(fix_pillow=False):
    import subprocess
    if fix_pillow:
        print("Detected Pillow incompatibility. Reinstalling stable Pillow...")
        subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "Pillow==10.4.0"], check=True)

    print("Aligning torch/torchaudio with CUDA 13.0 nightly builds...")
    # Target the specific nightly index for Colab's current CUDA 13.0 environment
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "torch", "torchaudio", "--index-url", "https://download.pytorch.org/whl/nightly/cu130"]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print("Successfully installed aligned versions.")
        print("IMPORTANT: Please go to 'Runtime' -> 'Restart session' to apply changes, then run this cell again.")
    else:
        print(f"Alignment failed with error:\n{result.stderr}")

try:
    from transformers import AutoProcessor
    # Qwen2/3-VL AutoProcessor triggers checks for the audio stack and image processing
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

    print(f"Processor class: {type(processor).__name__}")
    print(f"Processor summary: {processor}")
except (ImportError, RuntimeError, ModuleNotFoundError) as e:
    err_msg = str(e)
    fix_pillow = "_Ink" in err_msg or "PIL" in err_msg
    if "different CUDA versions" in err_msg or "AutoProcessor" in err_msg or fix_pillow:
        run_alignment(fix_pillow=fix_pillow)
    else:
        raise e

Detected Pillow incompatibility. Reinstalling stable Pillow...
Aligning torch/torchaudio with CUDA 13.0 nightly builds...
IMPORTANT: Please go to 'Runtime' -> 'Restart session' to apply changes, then run this cell again.


## Environment summary

A single collected record of the reproducibility-relevant facts gathered above. This is a
plain printed dict for at-a-glance review while running this notebook — not a persistence or
logging framework (out of scope for Phase 1; YAGNI).

In [9]:
import platform
import torch
import transformers
import datasets
import PIL

# Handle cases where the processor failed to load due to environment repairs
processor_class_name = type(processor).__name__ if 'processor' in globals() else "Not initialized (restart required)"

environment_summary = {
    "in_colab": IN_COLAB,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "datasets_version": datasets.__version__,
    "pillow_version": PIL.__version__,
    "cuda_available": cuda_available,
    "cuda_version": torch.version.cuda if cuda_available else None,
    "gpu_device_name": device_name,
    "gpu_total_memory_gib": total_memory_gib,
    "model_id": MODEL_ID,
    "model_config_class": type(model_config).__name__,
    "processor_class": processor_class_name,
}

for key, value in environment_summary.items():
    print(f"{key}: {value}")

in_colab: True
python_version: 3.12.13
torch_version: 2.13.0+cu130
transformers_version: 5.15.0
datasets_version: 5.0.1
pillow_version: 10.4.0
cuda_available: True
cuda_version: 13.0
gpu_device_name: Tesla T4
gpu_total_memory_gib: 14.56317138671875
model_id: Qwen/Qwen3-VL-4B-Instruct
model_config_class: Qwen3VLConfig
processor_class: Not initialized (restart required)
